## Практика. Регуляризация ближайших соседей

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.preprocessing import PolynomialFeatures


from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import BayesianRidge

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import confusion_matrix

>#### Задание 1
Прочитайте файл `PCB.csv`.

In [15]:
df = pd.read_csv('data/PCB.csv')
df

,copper_thickness,dielectric_thickness,dielectric_const,loss_tangent,solder_mask_thickness,solder_mask_viscosity,exposure_time,lamination_temp,lamination_pressure,etching_speed,...,pore_count,mask_peeling,contour_deviation,etching_uniformity,press_time,press_temperature,trace_width_min,layer_alignment,impedance_deviation,pcb_class
0,34.609494,0.193419,4.314433,0.003885,35.711230,3.341993,31.808574,159.038623,2.580114,2.178660,...,3.389508,0.980597,0.263658,91.707869,23.347714,197.438945,180.025809,105.071582,3.770046,0
1,28.489511,0.186562,4.442082,0.011333,28.632470,2.876198,37.069532,163.092805,2.961799,2.231623,...,1.175449,0.979452,0.342740,81.907522,23.182241,159.347050,174.248794,76.473399,4.143054,1
2,35.382828,0.168658,3.446464,0.007295,30.816347,2.480812,29.741620,150.141260,2.885362,2.257490,...,2.025417,1.122791,0.102606,80.929261,18.589156,149.653407,184.867630,105.872710,5.140972,0
3,33.605937,0.178395,3.033384,0.004317,23.642362,3.813067,26.724639,167.227230,2.196074,3.015834,...,2.774330,0.777289,0.230204,89.933775,19.105061,165.352761,167.652340,131.416337,3.391059,0
4,30.396843,0.198330,3.970997,0.002333,18.264830,3.425525,26.716428,112.175299,1.724289,1.837104,...,1.607913,0.649051,0.270953,84.944006,13.411430,167.464412,171.745848,109.765034,0.619956,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,31.820592,0.204175,3.334781,0.015207,33.837463,3.837299,34.848194,170.563331,2.310847,2.045401,...,0.753563,0.959163,0.213136,76.382162,24.075616,185.699519,168.115815,128.306619,7.303876,0
796,35.275889,0.237324,3.476749,0.004741,25.071925,4.086375,26.201135,188.633738,2.167253,2.412773,...,0.106078,0.076064,0.183505,101.401074,20.461466,163.997613,172.718466,141.629987,3.123223,0
797,28.188559,0.171152,2.871410,0.012863,17.389244,3.204306,28.275486,181.690820,1.853748,2.170355,...,2.275443,0.523878,0.151439,87.684568,26.230449,178.370886,151.007661,128.442918,4.387828,0
798,26.887256,0.211814,4.200254,0.009398,25.547348,2.791230,31.705009,133.871099,2.091725,2.035183,...,1.563210,1.043333,-0.002492,78.770858,12.541970,190.728220,154.608956,121.971319,3.627503,0


#### Контекст

Завод по производству многослойных печатных плат выпускает продукцию для двух сегментов рынка: промышленная электроника общего назначения и высокочастотная телекоммуникационная аппаратура. Каждая партия плат после изготовления проходит многоступенчатый контроль качества: измеряются геометрические параметры, электрические характеристики, свойства материалов и параметры технологического процесса.

Готовые платы сортируются на два класса точности:

- Класс 3 — для стандартной электроники (допуски по ширине проводников ±20%)
- Класс 4 — для высокочастотных применений (допуски ±15%)

Оба класса — качественная продукция, а не брак. Доли плат этих классов в производстве примерно равны. Брак отбраковывается автоматически.

#### Предикторы (все — float)

- trace_width_min — минимальная ширина проводника, мкм,
- layer_alignment — точность совмещения слоёв, мкм,
- impedance_deviation — отклонение волнового сопротивления от номинала, Ом,
- copper_thickness — толщина медного слоя, мкм,
- dielectric_thickness — толщина диэлектрика, мкм,
- dielectric_const — диэлектрическая проницаемость,
- loss_tangent — тангенс угла диэлектрических потерь,
- solder_mask_thickness — толщина защитной маски, мкм,
- solder_mask_viscosity — вязкость паяльной маски, Па·с,
- exposure_time — время экспонирования, с,
- lamination_temp — температура ламинации, °C,
- lamination_pressure — давление ламинации, МПа,
- etching_speed — скорость травления, м/мин,
- etching_temp — температура травителя, °C,
- etching_concentration — концентрация травителя, г/л,
- drill_speed — скорость сверления, мм/мин,
- drill_diameter — диаметр сверла, мм,
- drill_temperature — температура сверления, °C,
- electroless_time — время осаждения химической меди, мин,
- electroless_temp — температура осаждения химической меди, °C,
- plating_ph — pH ванны гальваники,
- plating_current — плотность тока гальваники, А/дм²,
- surface_roughness — шероховатость поверхности Ra, мкм,
- surface_gloss — блеск поверхности, %,
- pore_count — количество пор в фольге, шт/дм²,
- mask_peeling — отслоение маски, мм²	float
- contour_deviation — отклонение от контура платы, мм,
- etching_uniformity — равномерность травления, %,
- press_time — время опрессовки, мин,
- press_temperature — температура прессования, °C.

#### Таргет

- pcb_class — класс точности (1 — 4-й класс, 0 — 3-й класс) (int).

#### Бизнес-задача

Построить модель, которая по параметрам производственного контроля предскажет класс точности платы.

>#### Задание 2
Обучите baseline-модель бинарной классификации.

In [3]:
# Код

>#### Задание 3
Подберите оптимальное число ближайших соседей. Заметно ли вырос скор?

In [1]:
# Код

>#### Задание 4
Проведите нормализацию данных и еще раз подберите оптимальное число ближайших соседей. Заметно ли вырос скор? А по сравнению с baseline-моделью?

In [2]:
# Код

>#### Задание 5
Подберите оптимальное значение показателя метрики Минковского, после чего, уже в этой метрике — еще раз подберите оптимальное число ближайших соседей. Заметно  ли вырос скор? А по сравнению с baseline-моделью?

In [3]:
# Код

>#### Задание 6
Определите сигнальные предикторы. Удалите шумовые предикторы и проведите регуляризаци классификатора на сигнальных. Сравните результаты с полученными выше.

In [4]:
# Код

## Домашнее задание

>#### Задание 1
Прочитайте файл `tuning.csv`.

In [8]:
df = pd.read_csv('data/tuning.csv')
df

,mean_engine_rpm,max_engine_rpm,mean_speed,max_speed,mean_trip_duration,mean_trip_distance,n_trips_per_day,mean_coolant_temp,mean_intake_temp,mean_oil_temp,...,n_hard_accelerations,n_hard_brakings,mean_lambda_sensor_1,mean_lambda_sensor_2,mean_alternator_voltage,mean_knock_retard,max_knock_retard,max_intake_pressure,max_coolant_temp,is_tuned
0,2621.886832,5638.439216,63.798068,186.753502,28.336023,24.115324,3.035319,89.273395,24.420092,105.090119,...,28.997710,33.522611,0.751880,0.678267,14.410536,-1.671737,5.209816,144.718687,113.531272,0
1,2084.006358,5467.297831,73.428276,178.399711,37.363432,32.789482,3.590354,94.280975,46.113177,112.636753,...,23.689083,17.835006,0.664165,0.568872,14.483821,-0.376140,6.257196,169.866734,109.130655,0
2,2800.180478,6347.207859,64.099819,205.921309,37.158294,4.332753,1.638866,98.913838,32.214070,94.649670,...,39.291031,57.386499,0.529611,0.752767,14.483974,-3.990013,3.494602,194.923109,113.945046,1
3,2876.225887,6201.483532,93.582475,198.201995,20.490432,18.576422,1.418370,90.718613,34.425407,95.636375,...,64.239806,33.286907,0.693004,0.499073,14.336973,-0.848459,-1.375821,167.324543,107.714233,0
4,1719.585925,4876.166762,81.446812,90.425324,26.940174,31.949840,3.773447,89.973711,37.268281,98.329066,...,48.428484,42.416559,0.581558,0.431915,14.765339,-0.339627,3.486534,184.892297,108.335597,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,2656.804375,5456.969461,37.952137,164.827737,41.201061,25.391905,4.261945,89.059624,34.329450,114.118630,...,47.351119,14.926338,0.705861,0.679357,14.719389,-1.710603,3.790417,153.989166,106.430972,0
1996,2621.534970,6809.328331,63.358391,241.214802,24.368202,20.453101,3.860565,95.816098,37.726560,93.503050,...,33.582785,32.582348,0.731183,0.639799,13.931951,-3.726878,4.624317,215.642497,116.325599,1
1997,2463.383992,5803.423381,84.201614,140.395961,25.852488,32.049722,3.913610,92.852838,36.305883,105.130497,...,43.067386,18.196889,0.629498,0.567062,14.168511,-3.339735,5.214591,147.221218,107.431399,0
1998,2287.671269,5480.317032,63.047497,192.777669,36.247442,10.798930,3.331150,88.650938,21.315671,104.248342,...,55.413167,51.052987,0.678521,0.568906,13.887457,-2.706421,5.344313,209.388189,108.101109,1


#### Контекст

Крупный автопроизводитель собирает телематические данные с автомобилей, проданных год назад. У каждой машины есть бортовой компьютер, который передаёт на сервер производителя показания десятков датчиков при каждом запуске двигателя и в движении. Часть владельцев (примерно половина) неофициально модифицировала свои автомобили: перепрограммировала блок управления двигателем (чип-тюнинг), заменила выпускную систему, установила спортивные воздушные фильтры. Формально эти изменения не зарегистрированы — владельцы не сообщают о них, так как это лишает их заводской гарантии и может противоречить законодательству страны.

#### Предикторы

Все величины — средние значения за последние 30 дней эксплуатации. 

- mean_engine_rpm— средние обороты двигателя, об/мин,
- max_engine_rpm— максимальные обороты двигателя за месяц, об/мин,
- mean_coolant_temp — средняя температура охлаждающей жидкости, °C,
- max_coolant_temp — максимальная температура охлаждающей жидкости, °C,
- mean_intake_temp — средняя температура воздуха на впуске, °C,
- mean_intake_pressure — среднее давление наддува (для турбомоторов), кПа,
- max_intake_pressure — максимальное давление наддува, кПа,
- mean_oil_temp — средняя температура масла, °C,
- mean_oil_pressure — среднее давление масла, кПа,
- mean_exhaust_temp — средняя температура выхлопных газов перед катализатором, °C,
- max_exhaust_temp — максимальная температура выхлопа, °C,
- mean_fuel_trim_long — долгосрочная коррекция топливоподачи, %,
- mean_fuel_trim_short — краткосрочная коррекция топливоподачи, %,
- mean_ignition_timing — средний угол опережения зажигания, °,
- max_ignition_timing — максимальный угол опережения зажигания, °,
- mean_throttle_pos — среднее положение дроссельной заслонки, %,
- max_throttle_pos — максимальное положение дроссельной заслонки, %,
- mean_boost_target — целевое давление наддува, кПа,
- mean_maf_rate — средний расход воздуха через датчик MAF, г/с,
- max_maf_rate — максимальный расход воздуха через MAF, г/с,
- mean_knock_retard — среднее значение детонационной коррекции, °,
- max_knock_retard — максимальная детонационная коррекция, °,
- mean_speed — средняя скорость движения, км/ч,
- max_speed — максимальная скорость, км/ч,
- mean_trip_duration — средняя длительность поездки, мин,
- mean_trip_distance — средняя длина поездки, км,
- n_trips_per_day — среднее число поездок в день,
- mean_engine_load — средняя нагрузка на двигатель, %,
- max_engine_load — максимальная нагрузка на двигатель, %,
- mean_afr — среднее соотношение воздух/топливо (λ),
- mean_fuel_consumption — средний расход топлива, л/100 км,
- mean_accel_pedal_pos — среднее положение педали акселератора, %,
- max_accel_pedal_pos — максимальное положение педали акселератора, %,
- mean_brake_pedal_pos — среднее положение педали тормоза, %,
- n_hard_accelerations — число резких ускорений за месяц,
- n_hard_brakings — число резких торможений за месяц,
- mean_lambda_sensor_1 — средний сигнал датчика кислорода №1, В,
- mean_lambda_sensor_2 — средний сигнал датчика кислорода №2, В,
- mean_fuel_pressure — среднее давление в топливной рампе, кПа,
- ean_alternator_voltage — среднее напряжение бортовой сети, В.

#### Таргет

- is_tuned — бинарный признак: 1 — автомобиль модифицирован (тюнинг), 0 — стандартный. Неизвестен производителю напрямую, восстанавливается по телематическим данным.

#### Бинес-задача

Автопроизводитель не знает, какие именно машины из когорты были модифицированы, но подозревает, что такие есть. Для него критически важно отличать их от стандартных: во-первых, чтобы снимать с гарантии двигатели, вышедшие из строя из-за тюнинга; во-вторых, для корректного планирования гарантийных резервов и сервисных работ.

>#### Задание 2
Обучите baseline-модель бинарной классификации.

In [9]:
# Код

>#### Задание 3
Подберите оптимальное число ближайших соседей. Заметно ли вырос скор?

In [10]:
# Код

>#### Задание 4
Проведите нормализацию данных и еще раз подберите оптимальное число ближайших соседей. Заметно ли вырос скор? А по сравнению с baseline-моделью?

In [11]:
# Код

>#### Задание 5
Подберите оптимальное значение показателя метрики Минковского, после чего, уже в этой метрике — еще раз подберите оптимальное число ближайших соседей. Заметно  ли вырос скор? А по сравнению с baseline-моделью?

In [12]:
# Код

>#### Задание 6
Определите сигнальные предикторы. Удалите шумовые предикторы и проведите регуляризаци классификатора на сигнальных. Сравните результаты с полученными выше.

In [13]:
# Код